# RAG-Powered FAQ Agent: grounded, measured and privacy-aware

Answers customer questions from a company FAQ using retrieval-augmented generation, and **proves how well it works** with an evaluation scorecard.

| Feature | What it does |
|---|---|
| **One chunk per FAQ entry** | The FAQ is split into its question–answer pairs, each with an ID (`FAQ-05`), so retrieval is precise and every answer cites exactly which entry it used |
| **Refusal decided in code** | If no FAQ entry is similar enough to the question, the agent declines **without calling the LLM**. The threshold is calibrated on separate questions, not hard-coded by guesswork |
| **PII masking** | Emails, phone numbers and card numbers (checked with the Luhn algorithm) are masked in the question before it reaches retrieval or the LLM |
| **Evaluation scorecard** | 18 labelled questions (paraphrases, detail traps, out-of-scope questions and a PII case) are scored on retrieval, refusals, answer correctness and citations |

**Stack:** local `all-MiniLM-L6-v2` embeddings (free, no key) · FAISS · OpenAI or Anthropic for answers. No Azure needed.

**Setup:**
1. Add `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` to Colab **Secrets** (🔑) with notebook access on.
2. Upload `faq.txt` to the Colab file browser.
3. Set `PROVIDER` in Step 6.

Step 4 (offline tests) needs no API key.

## Step 1: Install packages

In [1]:
!pip install -q -U langchain-core langchain-community langchain-huggingface sentence-transformers faiss-cpu langchain-openai langchain-anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.5/862.5 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sour

## Step 2: Load the FAQ as one chunk per entry

A generic splitter cuts by character count and can separate a question from its answer. This FAQ has a clear structure, so each `Q:`/`A:` pair becomes one document with its own ID.

In [2]:
import os
import re
from dataclasses import dataclass, field

from langchain_core.documents import Document


@dataclass
class FAQEntry:
  faq_id: str
  question: str
  answer: str


def parse_faq(text: str) -> list:
  """Split 'Q: ... / A: ...' blocks into entries. Handles Windows line endings and multi-line answers."""
  text = text.replace("\r\n", "\n").replace("\r", "\n")
  pairs = re.findall(r"^Q:\s*(.+?)\s*\nA:\s*(.+?)(?=\n\s*\nQ:|\nQ:|\Z)", text, flags=re.MULTILINE | re.DOTALL)
  return [FAQEntry(f"FAQ-{i:02d}", q.strip(), " ".join(a.split())) for i, (q, a) in enumerate(pairs, 1)]


def load_faq(path: str = "faq.txt") -> list:
  with open(path, encoding="utf-8") as f:
    return parse_faq(f.read())

## Step 3: PII masking, the refusal rule and the scoring logic

These are plain Python functions with no LLM involved, so Step 4 can test them offline.

- **PII:** card numbers are only masked if they pass the **Luhn check** that real card numbers satisfy, and phone numbers must have 10–12 digits. That way an order number isn't mistaken for either. (The phone pattern is tuned for North American formats.)
- **Refusal:** `should_abstain` compares the best similarity score with `MIN_SIMILARITY`.
- **Scoring:** `score_case` grades one answer against its labelled expectation.

In [3]:
# ---------- PII masking ----------
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
CARD_CANDIDATE_RE = re.compile(r"\b(?:\d[ -]?){12,18}\d\b")
PHONE_RE = re.compile(r"(?<![\w+])(?:\+?\d{1,3}[-.\s]?)?(?:\(\d{3}\)|\d{3})[-.\s]?\d{3}[-.\s]?\d{4}(?!\w)")


def luhn_valid(digits: str) -> bool:
    total, double = 0, False
    for d in reversed(digits):
        n = int(d) * (2 if double else 1)
        total += n - 9 if n > 9 else n
        double = not double
    return total % 10 == 0


def mask_pii(text: str):
    """Return (masked_text, list of PII types found). Cards are checked first, so their digits aren't read as a phone number."""
    found = []

    def card(match):
        digits = re.sub(r"\D", "", match.group(0))
        if 13 <= len(digits) <= 19 and luhn_valid(digits):
            found.append("CARD")
            return "[CARD]"
        return match.group(0)            # not a real card number (e.g. an order ID): leave it

    def tag(label):
        def repl(_match):
            found.append(label)
            return f"[{label}]"
        return repl

    def phone(match):
        digits = re.sub(r"\D", "", match.group(0))
        if 10 <= len(digits) <= 12:      # a longer run of digits is an ID, not a phone number
            found.append("PHONE")
            return "[PHONE]"
        return match.group(0)

    text = CARD_CANDIDATE_RE.sub(card, text)
    text = EMAIL_RE.sub(tag("EMAIL"), text)
    text = PHONE_RE.sub(phone, text)
    return text, found


# ---------- Refusal rule ----------
MIN_SIMILARITY = 0.40   # default; Step 8 calibrates it on separate questions
FALLBACK = "I couldn't find that in our FAQ. Please contact our support team, who can help directly."


def should_abstain(top_similarity: float, threshold: float) -> bool:
    return top_similarity < threshold


def suggest_threshold(in_scope: list, out_of_scope: list):
    """Midpoint between the weakest in-scope match and the strongest out-of-scope match.
    Returns (threshold, separable). If the groups overlap, there is no perfect threshold."""
    low_in, high_out = min(in_scope), max(out_of_scope)
    return round((low_in + high_out) / 2, 3), low_in > high_out


# ---------- Citations ----------
CITATION_RE = re.compile(r"\[(FAQ-\d{2})\]")


def check_citations(answer: str, provided_ids: list):
    """IDs the answer cites, and any it cites that were NOT in the retrieved context (a hallucinated source)."""
    cited = sorted(set(CITATION_RE.findall(answer)))
    return cited, [c for c in cited if c not in provided_ids]


# ---------- Scoring ----------
def normalise(text: str) -> str:
    return re.sub(r"[\s\-]", "", text.lower())


def score_case(case: dict, response) -> dict:
    """Grade one response. `must_include` is a list of groups; each group passes if ANY of its phrases appears."""
    expected = case["expected_faq"]
    in_scope = expected is not None
    result = {"id": case["id"], "type": case["type"],
              "abstain_correct": response.abstained == (not in_scope)}
    if in_scope:
        answer = normalise(response.answer)
        result["retrieval_hit"] = expected in response.retrieved_ids
        result["answer_correct"] = (not response.abstained) and all(
            any(normalise(p) in answer for p in group) for group in case["must_include"])
        result["cited_correctly"] = expected in response.cited_ids
    if case.get("pii"):
        result["pii_masked"] = sorted(set(response.pii_found)) == sorted(case["pii"])
    result["passed"] = all(v for k, v in result.items() if k not in ("id", "type"))
    return result

## Step 4: Offline tests (no API key, no model download)

These check the parsing, PII masking, refusal rule, citation check and scoring before any model runs.

In [4]:
@dataclass
class FakeResponse:   # same fields as the real Response in Step 7
    answer: str = ""
    abstained: bool = False
    retrieved_ids: list = field(default_factory=list)
    cited_ids: list = field(default_factory=list)
    pii_found: list = field(default_factory=list)

# Parsing: Windows line endings and a multi-line answer
sample = "Q: First question?\r\nA: First answer.\r\n\r\nQ: Second question?\r\nA: Second answer\r\ncontinues here.\r\n"
entries = parse_faq(sample)
assert [e.faq_id for e in entries] == ["FAQ-01", "FAQ-02"]
assert entries[1].answer == "Second answer continues here."
print("✓ parsing: one entry per Q/A pair, CRLF and multi-line answers handled")

# PII: a real-format Amex test card, email and phone are masked; an order number is not
masked, found = mask_pii("Email jane.doe@gmail.com, card 3782 822463 10005, call +1-415-555-0199, order 1234567890123.")
assert "[EMAIL]" in masked and "[CARD]" in masked and "[PHONE]" in masked
assert "1234567890123" in masked             # fails the Luhn check, so it isn't treated as a card
assert sorted(found) == ["CARD", "EMAIL", "PHONE"]
print("✓ PII: card (Luhn-checked), email and phone masked; order number left alone")

# Refusal rule and threshold suggestion
assert should_abstain(0.31, 0.40) and not should_abstain(0.55, 0.40)
assert suggest_threshold([0.62, 0.48, 0.71], [0.22, 0.30]) == (0.39, True)
assert suggest_threshold([0.62, 0.28], [0.35, 0.20])[1] is False     # overlapping groups are reported
print("✓ refusal: below-threshold questions abstain; overlap between groups is detected")

# Citations: a cited ID that wasn't retrieved is caught
assert check_citations("30 days [FAQ-01], see also [FAQ-09].", ["FAQ-01", "FAQ-05"]) == (["FAQ-01", "FAQ-09"], ["FAQ-09"])
print("✓ citations: a source that wasn't retrieved is flagged")

# Scoring: correct answer, wrong answer, correct refusal, wrong refusal
case = {"id": 1, "type": "paraphrase", "expected_faq": "FAQ-05", "must_include": [["2 hours", "two hours"]]}
good = FakeResponse("Only within 2 hours of placement [FAQ-05].", False, ["FAQ-05"], ["FAQ-05"])
bad = FakeResponse("Yes, any time [FAQ-05].", False, ["FAQ-05"], ["FAQ-05"])
assert score_case(case, good)["passed"] and not score_case(case, bad)["passed"]
oos = {"id": 2, "type": "out_of_scope", "expected_faq": None}
assert score_case(oos, FakeResponse(FALLBACK, True))["passed"]
assert not score_case(oos, FakeResponse("Yes, 10% off!", False))["passed"]
print("✓ scoring: right and wrong answers and refusals graded correctly")
print("\nAll offline tests passed.")

✓ parsing: one entry per Q/A pair, CRLF and multi-line answers handled
✓ PII: card (Luhn-checked), email and phone masked; order number left alone
✓ refusal: below-threshold questions abstain; overlap between groups is detected
✓ citations: a source that wasn't retrieved is flagged
✓ scoring: right and wrong answers and refusals graded correctly

All offline tests passed.


## Step 5: Embed the FAQ entries and build the FAISS index

Embeddings are normalised, so the dot product of two vectors is their **cosine similarity** (closer to 1 means more similar). The entry vectors are kept so each result gets an exact similarity score.

In [6]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

FAQ = load_faq("faq.txt")
ENTRY_BY_ID = {e.faq_id: e for e in FAQ}
docs = [Document(page_content=f"Q: {e.question}\nA: {e.answer}", metadata={"faq_id": e.faq_id}) for e in FAQ]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", encode_kwargs={"normalize_embeddings": True})
vectorstore = FAISS.from_documents(docs, embeddings)
ENTRY_VECTORS = {d.metadata["faq_id"]: np.array(v) for d, v in zip(docs, embeddings.embed_documents([d.page_content for d in docs]))}

print(f"Indexed {len(FAQ)} FAQ entries:")
for e in FAQ:
    print(f"  {e.faq_id}: {e.question}")


def retrieve(query: str, k: int = 3) -> list:
  """Top-k FAQ entries with their cosine similarity to the query, highest first."""
  q = np.array(embeddings.embed_query(query))
  hits = vectorstore.similarity_search(query, k=k)
  scored = [(ENTRY_BY_ID[d.metadata["faq_id"]], float(q @ ENTRY_VECTORS[d.metadata["faq_id"]])) for d in hits]
  return sorted(scored, key=lambda pair: -pair[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 10 FAQ entries:
  FAQ-01: What is your return policy?
  FAQ-02: Do you offer international shipping?
  FAQ-03: How can I contact customer support?
  FAQ-04: What payment methods are accepted?
  FAQ-05: How can I cancel my order?
  FAQ-06: Is there a warranty on products?
  FAQ-07: Can I track my order?
  FAQ-08: What should I do if I receive a damaged product?
  FAQ-09: Are there any discounts for bulk purchases?
  FAQ-10: Do you offer gift wrapping?


## Step 6: Connect to the LLM
Set `PROVIDER` to `"openai"` or `"anthropic"`, matching the key in your Secrets.

In [7]:
def _load(key):
  """Colab Secrets first, then an environment variable."""
  try:
    from google.colab import userdata
    value = userdata.get(key)
    if value:
      return value
  except Exception:
    pass
  return os.environ.get(key)

PROVIDER = "openai"   # or "anthropic"

if PROVIDER == "openai":
  from langchain_openai import ChatOpenAI
  llm = ChatOpenAI(model="gpt-5-mini", api_key=_load("OPENAI_API_KEY"))
elif PROVIDER == "anthropic":
  from langchain_anthropic import ChatAnthropic
  llm = ChatAnthropic(model="claude-haiku-5-5", api_key=_load("ANTHROPIC_API_KEY"), max_tokens=1024)
else:
  raise ValueError('PROVIDER must be "openai" or "anthropic"')

print(f"Using {PROVIDER}")

Using openai


## Step 7: The answer pipeline

1. Mask PII in the question.
2. Retrieve the top FAQ entries with their similarity scores.
3. **If the best score is below the threshold, decline without calling the LLM.**
4. Otherwise, answer only from the retrieved entries, citing them as `[FAQ-xx]`.
5. Check that every cited ID was actually retrieved.

In [8]:
@dataclass
class Response:
  question: str
  masked_question: str
  answer: str
  abstained: bool
  top_similarity: float
  retrieved_ids: list
  cited_ids: list
  pii_found: list
  llm_called: bool
  unsupported_citations: list = field(default_factory=list)


PROMPT = """You are a customer support assistant. Answer using ONLY the FAQ entries below.
Cite every entry you use with its ID in square brackets, e.g. [FAQ-05].
If the entries don't answer the question, reply exactly: "{fallback}"
Keep the answer to one or two sentences.

FAQ entries:
{context}

Customer question: {question}
Answer:"""


def text_of(message) -> str:
  """OpenAI returns a string; Anthropic can return a list of content blocks."""
  if isinstance(message.content, str):
    return message.content
  return "".join(b.get("text", "") for b in message.content if isinstance(b, dict))


def answer(question: str, threshold: float | None = None, k: int = 3) -> Response:
  threshold = MIN_SIMILARITY if threshold is None else threshold
  masked, pii = mask_pii(question)
  hits = retrieve(masked, k=k)
  top = hits[0][1] if hits else 0.0
  retrieved = [e.faq_id for e, _ in hits]

  if should_abstain(top, threshold):        # decided in code: no LLM call, no cost, no guessing
    return Response(question, masked, FALLBACK, True, top, retrieved, [], pii, llm_called=False)

  usable = [(e, s) for e, s in hits if s >= threshold]
  context = "\n".join(f"[{e.faq_id}] Q: {e.question}\nA: {e.answer}" for e, _ in usable)
  reply = text_of(llm.invoke(PROMPT.format(fallback=FALLBACK, context=context, question=masked))).strip()

  cited, unsupported = check_citations(reply, [e.faq_id for e, _ in usable])
  declined = reply.startswith(FALLBACK[:30])
  return Response(question, masked, reply, declined, top, retrieved, cited, pii, llm_called=True, unsupported_citations=unsupported)


def show(r: Response):
  print(f"Q: {r.question}")
  if r.pii_found:
    print(f"   (masked {', '.join(sorted(set(r.pii_found)))} → {r.masked_question})")
  print(f"A: {r.answer}")
  print(f"   top similarity {r.top_similarity:.2f} | LLM called: {r.llm_called} | cited: {r.cited_ids or '-'}")
  if r.unsupported_citations:
    print(f"   ⚠ cited sources that weren't retrieved: {r.unsupported_citations}")

## Step 8: Calibrate the refusal threshold

The threshold is set from a **separate calibration set**, not from the evaluation questions, so the scorecard in Step 9 stays a fair test.

In [9]:
CALIBRATION_IN = [
  "What's your refund window?",
  "Which countries do you ship to?",
  "What cards can I pay with?",
  "Does my laptop come with a warranty?",
  "Will I get a tracking number?",
]
CALIBRATION_OUT = [
  "What's the weather like today?",
  "Do you have a mobile app?",
  "Who founded the company?",
]

in_scores = [retrieve(q, k=1)[0][1] for q in CALIBRATION_IN]
out_scores = [retrieve(q, k=1)[0][1] for q in CALIBRATION_OUT]
for q, s in zip(CALIBRATION_IN + CALIBRATION_OUT, in_scores + out_scores):
  print(f"  {s:.2f}  {'in ' if q in CALIBRATION_IN else 'OUT'}  {q}")

MIN_SIMILARITY, separable = suggest_threshold(in_scores, out_scores)
print(f"\nMIN_SIMILARITY set to {MIN_SIMILARITY} "
      + ("(groups separate cleanly)" if separable else "(groups OVERLAP: some errors are unavoidable at any threshold)"))

  0.39  in   What's your refund window?
  0.68  in   Which countries do you ship to?
  0.48  in   What cards can I pay with?
  0.57  in   Does my laptop come with a warranty?
  0.66  in   Will I get a tracking number?
  0.05  OUT  What's the weather like today?
  0.27  OUT  Do you have a mobile app?
  0.10  OUT  Who founded the company?

MIN_SIMILARITY set to 0.329 (groups separate cleanly)


## Step 9: Evaluation scorecard

18 labelled cases:

| Type | What it tests |
|---|---|
| `paraphrase` | Different wording from the FAQ, e.g. "Can I send stuff back?" |
| `detail` | The correct answer depends on a detail, e.g. an *opened* item or an order placed 3 hours ago |
| `out_of_scope` | Not in the FAQ: the agent should decline |
| `pii` | Personal data in the question must be masked, and the answer must still be right |

In [14]:
EVAL_SET = [
  {"id": 1,  "type": "paraphrase", "q": "Can I send back something I bought?", "expected_faq": "FAQ-01", "must_include": [["30"]]},
  {"id": 2,  "type": "detail", "q": "Can I return an item I've already opened?", "expected_faq": "FAQ-01", "must_include": [["unopened"]]},
  {"id": 3,  "type": "paraphrase", "q": "Do you deliver to other countries?", "expected_faq": "FAQ-02", "must_include": [["50"]]},
  {"id": 4,  "type": "paraphrase", "q": "What's the support phone number?", "expected_faq": "FAQ-03", "must_include": [["800-123-4567", "8001234567"]]},
  {"id": 5,  "type": "detail", "q": "Can I reach support on a Saturday?", "expected_faq": "FAQ-03", "must_include": [["monday", "mon"], ["friday", "fri"]]},
  {"id": 6,  "type": "paraphrase", "q": "Can I pay with American Express?", "expected_faq": "FAQ-04", "must_include": [["american express", "amex"]]},
  {"id": 7,  "type": "paraphrase", "q": "Is UPI accepted?", "expected_faq": "FAQ-04", "must_include": [["upi"]]},
  {"id": 8,  "type": "detail", "q": "I placed an order 3 hours ago. Can I still cancel it?", "expected_faq": "FAQ-05", "must_include": [["2 hours", "two hours"]]},
  {"id": 9,  "type": "paraphrase", "q": "How long is the warranty on electronics?", "expected_faq": "FAQ-06", "must_include": [["1 year", "one year", "1-year"]]},
  {"id": 10, "type": "paraphrase", "q": "How do I find out where my package is?", "expected_faq": "FAQ-07", "must_include": [["track"]]},
  {"id": 11, "type": "detail", "q": "My item arrived broken last week. Can I still get a replacement?", "expected_faq": "FAQ-08", "must_include": [["48"]]},
  {"id": 12, "type": "paraphrase", "q": "Do you give discounts on large orders?", "expected_faq": "FAQ-09", "must_include": [["sales@example.com"]]},
  {"id": 13, "type": "paraphrase", "q": "How much does gift wrapping cost?", "expected_faq": "FAQ-10", "must_include": [["50"]]},
  {"id": 14, "type": "out_of_scope", "q": "Do you offer a student discount on enterprise plans?", "expected_faq": None},
  {"id": 15, "type": "out_of_scope", "q": "What's the name of your CEO?", "expected_faq": None},
  {"id": 16, "type": "out_of_scope", "q": "Can I pick up my order in a store?", "expected_faq": None},
  {"id": 17, "type": "out_of_scope", "q": "Do you price-match competitors?", "expected_faq": None},
  {"id": 18, "type": "pii", "q": "My email is jane.doe@gmail.com and my card is 3782 822463 10005. How do I cancel my order?","expected_faq": "FAQ-05", "must_include": [["2 hours", "two hours"]], "pii": ["CARD", "EMAIL"]},
]

results, responses = [], {}
for case in EVAL_SET:
  r = answer(case["q"])
  responses[case["id"]] = r
  results.append(score_case(case, r))


def rate(key, rows):
  rows = [x for x in rows if key in x]
  return f"{sum(x[key] for x in rows)}/{len(rows)}" if rows else "-"

in_scope = [x for x in results if "retrieval_hit" in x]
print("SCORECARD")
print(f"  Overall passed            {rate('passed', results)}")
print(f"  Retrieval hit (top 3)     {rate('retrieval_hit', in_scope)}")
print(f"  Answer correct            {rate('answer_correct', in_scope)}")
print(f"  Cited the right entry     {rate('cited_correctly', in_scope)}")
print(f"  Correct refusal decisions {rate('abstain_correct', results)}")
print(f"  PII masked                {rate('pii_masked', results)}")
print(f"  LLM calls saved by refusing in code: {sum(not r.llm_called for r in responses.values())}")

print("\nBy type:")
for t in ["paraphrase", "detail", "out_of_scope", "pii"]:
  print(f"  {t:<13} {rate('passed', [x for x in results if x['type'] == t])}")

failures = [x for x in results if not x["passed"]]
print(f"\nFailures ({len(failures)}):")
for x in failures:
  print("-" * 70)
  show(responses[x["id"]])
  print("   checks:", {k: v for k, v in x.items() if k not in ("id", "type", "passed")})

SCORECARD
  Overall passed            17/18
  Retrieval hit (top 3)     14/14
  Answer correct            13/14
  Cited the right entry     13/14
  Correct refusal decisions 17/18
  PII masked                1/1
  LLM calls saved by refusing in code: 2

By type:
  paraphrase    8/9
  detail        4/4
  out_of_scope  4/4
  pii           1/1

Failures (1):
----------------------------------------------------------------------
Q: How do I find out where my package is?
A: I couldn't find that in our FAQ. Please contact our support team, who can help directly.
   top similarity 0.31 | LLM called: False | cited: -
   checks: {'abstain_correct': False, 'retrieval_hit': True, 'answer_correct': False, 'cited_correctly': False}


## Failure analysis: 1 of 18 cases failed

**Result: 17/18 passed.** The one failure is a **false refusal**: the agent declined a question the FAQ can answer. Retrieval found the right entry; the refusal threshold stopped the answer.

## Why the similarity score was so low

The question and the FAQ entry mean the same thing but share almost no words:

| | Wording |
|---|---|
| **Question** | "How do I find out where my **package** is?" |
| **FAQ-07** | "Can I **track** my **order**? ... you will receive a **tracking link**..." |

*"Package"* vs. *"order"* and *"where is it"* vs. *"track"* are a **vocabulary mismatch**. `all-MiniLM-L6-v2` is a small, fast embedding model and doesn't connect those phrasings strongly, so FAQ-07 ranked first, but with a weak score.

This is the **threshold trade-off** in practice: a threshold strict enough to reject out-of-scope questions will sometimes reject a valid question that's worded differently.

## How to fix it

> ⚠️ **Not by lowering the threshold until this case passes.** That's tuning to the test data, and it could let out-of-scope questions through.

1. **Add alternative phrasings to each entry when indexing**
   For example, embed FAQ-07 with *"Also asked as: where is my package, delivery status, order location"*. This is cheap and targets the actual cause. An LLM could generate the variants once, at indexing time.

2. **Use a two-level threshold**
   - **Below a hard floor (e.g. 0.25):** decline in code. No LLM call.
   - **Grey zone (0.25 up to the threshold):** pass the retrieved entries to the LLM and let it decide. The prompt already tells it to decline if the context doesn't answer the question.

   Clear cases stay cheap and deterministic; borderline cases get judgement.

3. **Try a stronger embedding model**
   For example `all-mpnet-base-v2` or `BAAI/bge-small-en-v1.5`. Rerun the calibration and the scorecard to compare.



## Step 10: Try your own question

In [12]:
show(answer("Hi, I'm jane.doe@gmail.com. Do you ship to Canada and how long does it take?"))

Q: Hi, I'm jane.doe@gmail.com. Do you ship to Canada and how long does it take?
   (masked EMAIL → Hi, I'm [EMAIL]. Do you ship to Canada and how long does it take?)
A: Yes — we ship to Canada; delivery times and shipping costs vary by destination [FAQ-02].
   top similarity 0.51 | LLM called: True | cited: ['FAQ-02']


## Design Notes

- **Reading the scorecard:** a failure isn't a bug to hide. Check which stage failed. A retrieval miss suggests better chunk text or hybrid keyword + vector search. A wrong answer with the right entry points to the prompt. A wrong refusal points to the threshold.
- **Threshold trade-off:** a higher `MIN_SIMILARITY` gives fewer wrong answers but more unnecessary refusals. For customer support, declining and handing over to a person is usually the safer failure.
- **Next Steps:** hybrid search for exact terms (order IDs, product names), follow-up questions that use conversation history, and agent tools such as opening a support ticket.